# FitCheck Phase 3: calibration on 2x T4

Pushed with the Kaggle CLI from `calibrate/kaggle/`, whose `kernel-metadata.json` asks for GPU T4 x2 and internet. In the web UI the same settings are Accelerator **GPU T4 x2** and Internet **on**, run with **Save Version, then Save & Run All (Commit)**.

It installs pinned versions, checks the activation formulas against autograd on the T4 and on CPU, runs the 55-run DeepSpeed sweep, and leaves these in the output:

- `results.jsonl`: one line per run
- `saved_tensors_cuda.json` and `saved_tensors_cpu.json`: the saved-tensor checks
- `runs/`: per-run rank files and launcher logs, for debugging

Download `results.jsonl` and both `saved_tensors_*.json` files.

In [ ]:
# SMOKE = True: the environment checks, the DeepSpeed op build, both saved-tensor
# checks and 2 sweep runs (LoRA on ZeRO-2 with FusedAdam, ZeRO-3 with CPU Adam and both
# offloads), then stop. SMOKE = False: all 55 runs.
SMOKE = False
SMOKE_RUNS = "0.5B-z2-lora16-s1024-mb1-sdpa-g2 0.5B-z3-oo-op-s1024-mb1-sdpa-g2"
SWEEP_ARGS = f"--only {SMOKE_RUNS}" if SMOKE else ""
print("smoke run" if SMOKE else "full sweep")

In [ ]:
# The machine: two T4s, the driver, nvcc, Python, and host RAM (total and
# available). The 1.5B offload runs need the most host RAM, so they run last.
!nvidia-smi
!nvcc --version
!python --version
!free -h

In [ ]:
# Cloned outside /kaggle/working, so the code doesn't end up in the output.
!git clone --depth 1 --branch phase-3 https://github.com/shravanibnikam/fit-check.git /tmp/fit-check
!git -C /tmp/fit-check log --oneline -1

In [ ]:
# Pinned installs. DeepSpeed JIT-compiles FusedAdam with the local nvcc, so torch's
# CUDA major version must match nvcc's: the cu126 wheel for nvcc 12.x, cu130 for 13.x.
# Both include sm_75 (.ci/manywheel/build_env_setup.py in PyTorch v2.14.1).
import importlib.metadata
import re
import subprocess
import sys


def pip(*args):
    subprocess.run([sys.executable, "-m", "pip", *args], check=True)


# Kaggle's own torch (2.11.0+cu128 in image v172) is the fallback if DeepSpeed's
# ops don't build against 2.14.1.
preinstalled = importlib.metadata.version("torch")
with open("/kaggle/working/preinstalled_torch.txt", "w") as f:
    f.write(preinstalled)
print("preinstalled torch", preinstalled)

nvcc = subprocess.run(
    ["nvcc", "--version"], capture_output=True, text=True, check=True
).stdout
major = int(re.search(r"release (\d+)\.", nvcc).group(1))
index = {
    12: "https://download.pytorch.org/whl/cu126",
    13: "https://download.pytorch.org/whl/cu130",
}[major]
print("nvcc CUDA", major, "->", index)

# Kaggle's torchvision and torchaudio are built for its torch, and a mismatched
# torchvision can break transformers imports. Its torchao (0.10.0) is older than
# peft 0.21.2 accepts, and peft refuses to add LoRA adapters while it is
# installed (is_torchao_available in peft/import_utils.py).
pip("uninstall", "-y", "torchvision", "torchaudio", "torchao")
pip("install", "--quiet", "torch==2.14.1", "--index-url", index)
pip(
    "install",
    "--quiet",
    "deepspeed==0.19.9",
    "transformers==5.19.0",
    "peft==0.21.2",
    "accelerate==1.15.0",
    "-e",
    "/tmp/fit-check",
)

In [ ]:
# Build FusedAdam and DeepSpeedCPUAdam once, for the T4 only. If they don't build
# against torch 2.14.1, fall back to Kaggle's preinstalled torch and build again.
# Every run records the torch it used.
import os
import shutil

os.environ["TORCH_CUDA_ARCH_LIST"] = "7.5"
BUILD = (
    "import torch; from deepspeed.ops.adam import FusedAdam, DeepSpeedCPUAdam; "
    "FusedAdam([torch.nn.Parameter(torch.zeros(1, device='cuda'))]); "
    "DeepSpeedCPUAdam([torch.nn.Parameter(torch.zeros(1))]); print('ops built')"
)


def build():
    return subprocess.run([sys.executable, "-c", BUILD], check=False).returncode == 0


if not build():
    with open("/kaggle/working/preinstalled_torch.txt") as f:
        version, _, cuda = f.read().strip().partition("+")
    if not cuda:
        raise RuntimeError(f"can't tell the CUDA build of torch {version}")
    print(f"Falling back to torch {version} from whl/{cuda}")
    pip(
        "install",
        "--quiet",
        f"torch=={version}",
        "--index-url",
        f"https://download.pytorch.org/whl/{cuda}",
    )
    # Ops built for 2.14.1 can sit in the same cache folder; rebuild them.
    shutil.rmtree(os.path.expanduser("~/.cache/torch_extensions"), ignore_errors=True)
    if not build():
        raise RuntimeError(
            "DeepSpeed's ops don't build on either torch; send the output above"
        )

In [ ]:
# Stop here unless this is two T4s and torch is built for sm_75.
VERIFY = (
    "import torch, deepspeed, transformers, peft; "
    "print('torch', torch.__version__, 'cuda', torch.version.cuda, "
    "'deepspeed', deepspeed.__version__, 'transformers', transformers.__version__, "
    "'peft', peft.__version__); "
    "assert torch.cuda.device_count() == 2, torch.cuda.device_count(); "
    "assert torch.cuda.get_device_capability(0) == (7, 5), torch.cuda.get_device_name(0); "
    "assert 'sm_75' in torch.cuda.get_arch_list(), torch.cuda.get_arch_list(); "
    "print(torch.cuda.get_device_name(0), torch.cuda.get_arch_list())"
)
subprocess.run([sys.executable, "-c", VERIFY], check=True)

In [ ]:
# Both models at the revisions the sweep pins, downloaded once.
!python -c "from huggingface_hub import snapshot_download; snapshot_download('Qwen/Qwen2.5-0.5B', revision='060db6499f32faf8b98477b0a26969ef7d8b9987'); snapshot_download('Qwen/Qwen2.5-1.5B', revision='8faed761d45a263340a0528343f099c05c9a4323')"

In [ ]:
# The activation formulas against what autograd saves, on the same torch as the
# sweep: on the T4 (fp16, SDPA, so the memory-efficient kernel) and on CPU.
# The sweep runs either way, but fitcheck calibrate won't fit unless every CUDA
# case matches.
%cd /tmp/fit-check
!python scripts/saved_tensors.py --device cuda --json /kaggle/working/saved_tensors_cuda.json
!python scripts/saved_tensors.py --device cpu --json /kaggle/working/saved_tensors_cpu.json

In [ ]:
# The 55 runs, or the 2 smoke runs. Each prints its exact command first; a run
# that hangs is killed after 15 minutes. Rerunning this cell skips runs already in
# results.jsonl.
!TORCH_CUDA_ARCH_LIST=7.5 python calibrate/sweep.py --results /kaggle/working/results.jsonl --work-dir /kaggle/working/runs $SWEEP_ARGS

In [ ]:
import collections
import json

with open("/kaggle/working/results.jsonl") as f:
    rows = [json.loads(line) for line in f]
print(len(rows), "runs:", dict(collections.Counter(r["status"] for r in rows)))
print("torch used:", sorted({r["env"]["torch"] for r in rows if r.get("env")}))
few = [r["run"] for r in rows if r["status"] == "ok" and r["applied_steps"] < 4]
print("under 4 applied steps:", few)
for device in ("cuda", "cpu"):
    try:
        with open(f"/kaggle/working/saved_tensors_{device}.json") as f:
            check = json.load(f)
    except FileNotFoundError:
        print(f"saved tensors on {device}: no JSON written")
        continue
    bad = [f"{c['config']} {c['lora_rank']}" for c in check["cases"] if not c["ok"]]
    print(f"saved tensors on {device}: torch {check['torch']}, failed cases: {bad}")